# Seleção de Fontes para Enriquecimento

A base individual de alfabetização possui poucas variáveis disponíveis antes da aplicação da avaliação. Por isso, esta etapa busca fontes externas capazes de representar o contexto territorial, socioeconômico e educacional dos alunos.

As fontes são selecionadas considerando:

- possibilidade de integração por `id_municipio` ou `id_escola`; <br>
- cobertura dos anos necessários; <br>
- disponibilidade anterior ao momento da previsão; <br>
- relevância para o problema de alfabetização; <br>
- ausência de informações derivadas do próprio resultado da avaliação. <br>

Para reduzir risco de vazamento temporal, a primeira versão da base utilizará variáveis externas referentes ao ano anterior à avaliação.

## Fontes selecionadas

### Censo Escolar
Granularidade: escola e ano.
Será utilizado para representar características estruturais e educacionais da
escola, utilizando `id_escola` como chave de integração.

### População Municipal
Granularidade: município e ano.
Será utilizada para representar o porte populacional do município.

### PIB Municipal
Granularidade: município e ano.
Será utilizado para representar o contexto econômico municipal.


### Indicador de Nível Socioeconômico (INSE)
Granularidade: escola e edição do indicador.
O INSE será utilizado para representar o contexto socioeconômico dos alunos atendidos pela escola. O indicador considera aspectos relacionados à posse de bens e serviços no domicílio e à escolaridade dos pais ou responsáveis

# 1. Configuração
Nesta etapa são importadas as bibliotecas utilizadas no notebbok e feita a configuração para consumir as tabelas do Base de Dados

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

## Conexão Base-Dados

In [0]:
credencial_gcp = dbutils.secrets.get(
    scope="fiap-tech-challenge",
    key="gcp-service-account-b64"
)

if not credencial_gcp:
    raise ValueError("Credencial do Google Cloud não encontrada.")

print("Credencial carregada com sucesso.")

In [0]:
dbutils.widgets.text(
    "billing_project_id",
    "fiap-tech-challenge2",
    "Billing Project GCP"
)

billing_project_id = dbutils.widgets.get("billing_project_id")

print(f"Billing project: {billing_project_id}")

# 2. Tabela INSE

## Limitação da Integração por Escola

A base da Avaliação da Alfabetização disponibiliza o campo `id_escola` de forma mascarada. Esse identificador não corresponde ao código Inep utilizado em outras bases educacionais, como o Censo Escolar e o Indicador de Nível Socioeconômico (INSE).

Por esse motivo, não é possível realizar a integração direta das informações externas no nível da escola.

Para preservar informações educacionais e socioeconômicas relevantes, os indicadores externos serão agregados em níveis geográficos compatíveis com a base de alfabetização, principalmente por município e rede de ensino.

O campo `id_municipio`, por sua vez, permite a integração com fontes externas de contexto territorial e será utilizado como principal chave de enriquecimento.

## 2.1 Carregando a Base
A tabela de INSE por escola é carregada diretamente da Base dos Dados.

Como o identificador de escola da base de alfabetização é mascarado, os registros do INSE são agregados por id_municipio e rede, chaves disponíveis nas duas fontes.

É utilizada a edição de 2021, anterior às avaliações de alfabetização de 2023 e 2024. São consideradas apenas as redes Estadual (2) e Municipal (3), que compõem a população de modelagem.

O valor municipal do INSE é calculado como média ponderada pela quantidade de alunos considerados no indicador de cada escola.

In [0]:
# Parâmetros de conexão com o BigQuery
source_project = "basedosdados"
source_dataset_inse = "br_inep_indicador_nivel_socioeconomico"
source_table_inse = "escola"

tabela_inse = (
    f"{source_project}."
    f"{source_dataset_inse}."
    f"{source_table_inse}"
)


# Leitura da tabela física de INSE por escola
df_inse = (
    spark.read
    .format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load(tabela_inse)
)

df_inse.printSchema()

In [0]:
df_inse_2021 = df_inse.filter((F.col("ano") == 2021) & (F.col("rede").isin("2", "3")))

df_inse_2021_municipio_rede = df_inse_2021.groupBy("id_municipio", "rede").agg(
    (
        F.sum(F.col("inse") * F.col("quantidade_alunos_inse"))
        / F.sum("quantidade_alunos_inse")
    ).alias("inse_2021"),
    F.sum("quantidade_alunos_inse").alias("quantidade_alunos_inse_2021"),
    F.countDistinct("id_escola").alias("quantidade_escolas_inse_2021"),
)

## 2.2 Reconstrução da população
A população definida no diagnóstico inicial é reconstruída neste notebook para garantir sua execução de forma independente. <br>
São mantidos apenas alunos presentes, com prova preenchida e pertencentes às redes Estadual ou Municipal. <br>
Essa população será utilizada para avaliar a cobertura das fontes externas antes da construção da base analítica final.

In [0]:
# Leitura da base Silver de alunos
tabela_alunos = "workspace.silver.alunos"

df_alunos = spark.table(tabela_alunos)

# População elegível para modelagem
df_populacao_modelagem = df_alunos.filter(
    (F.col("presenca") == "1")
    & (F.col("preenchimento_caderno") == "1")
    & (F.col("rede").isin("2", "3"))
)

contextos_modelagem = df_populacao_modelagem.select(
    "ano", "id_municipio", "rede"
).distinct()

cobertura_inse = contextos_modelagem.join(
    df_inse_2021_municipio_rede, on=["id_municipio", "rede"], how="left"
)

## 2.3 Cobertura por município e rede
Antes de incorporar o indicador à base individual, é avaliada sua cobertura nas combinações de município e rede existentes na população de modelagem. <br>
Uma cobertura elevada indica que o INSE consegue representar a maior parte dos contextos educacionais presentes nas avaliações de 2023 e 2024.

In [0]:
display(
    cobertura_inse.groupBy("ano")
    .agg(
        F.count("*").alias("qtd_contextos"),
        F.sum(F.when(F.col("inse_2021").isNotNull(), 1).otherwise(0)).alias(
            "contextos_com_inse"
        ),
    )
    .withColumn(
        "pct_cobertura",
        F.round(F.col("contextos_com_inse") / F.col("qtd_contextos") * 100, 2),
    )
    .orderBy("ano")
)

## 2.4 Cobertura por aluno

In [0]:
df_populacao_com_inse = df_populacao_modelagem.join(
    df_inse_2021_municipio_rede, on=["id_municipio", "rede"], how="left"
)

display(
    df_populacao_com_inse.groupBy("ano")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.sum(F.when(F.col("inse_2021").isNotNull(), 1).otherwise(0)).alias(
            "alunos_com_inse"
        ),
    )
    .withColumn(
        "pct_cobertura",
        F.round(F.col("alunos_com_inse") / F.col("qtd_alunos") * 100, 2),
    )
    .orderBy("ano")
)

## 2.5 INSE por target
Como análise exploratória inicial, são comparados a média e a mediana do INSE entre alunos alfabetizados e não alfabetizados. <br>
O objetivo não é estabelecer causalidade, mas verificar se o contexto socioeconômico apresenta associação com o target antes da etapa de modelagem.

In [0]:
display(
    df_populacao_com_inse.filter(F.col("inse_2021").isNotNull())
    .groupBy("alfabetizado")
    .agg(
        F.round(F.avg("inse_2021"), 3).alias("inse_medio"),
        F.expr("percentile_approx(inse_2021, 0.5)").alias("inse_mediano"),
    )
    .orderBy("alfabetizado")
)

## 2.6 Taxa de Alfabetização por Faixa de INSE

A comparação entre médias apresenta apenas uma visão geral da relação entre o indicador e o target. <br>

Para explorar possíveis diferenças ao longo da distribuição, o INSE é dividido em faixas e calculada a taxa de alfabetização observada em cada grupo. <br>

As faixas são utilizadas exclusivamente para análise exploratória e não serão incorporadas como features na primeira versão do modelo.

In [0]:
df_inse_faixas = df_populacao_com_inse.filter(
    F.col("inse_2021").isNotNull()
).withColumn(
    "faixa_inse",
    F.when(F.col("inse_2021") < 4.4, "1 - Muito baixo")
    .when(F.col("inse_2021") < 4.7, "2 - Baixo")
    .when(F.col("inse_2021") < 5.0, "3 - Médio")
    .when(F.col("inse_2021") < 5.3, "4 - Alto")
    .otherwise("5 - Muito alto"),
)


display(
    df_inse_faixas.groupBy("faixa_inse")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.round(F.avg(F.col("alfabetizado").cast("double")) * 100, 2).alias(
            "taxa_alfabetizacao"
        ),
    )
    .orderBy("faixa_inse")
)

## 2.7 Análise por Decis do INSE

Como as faixas anteriores foram definidas apenas para exploração, é realizada uma segunda análise dividindo a população em dez grupos de tamanho semelhante de acordo com o valor do INSE.

A análise por decis permite observar com maior detalhe se existe um gradiente entre nível socioeconômico e taxa de alfabetização, sem depender de limites definidos manualmente.

In [0]:
window_inse = Window.orderBy("inse_2021")

df_inse_decis = df_populacao_com_inse.filter(F.col("inse_2021").isNotNull()).withColumn(
    "decil_inse", F.ntile(10).over(window_inse)
)

display(
    df_inse_decis.groupBy("decil_inse")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.round(F.avg(F.col("alfabetizado").cast("double")) * 100, 2).alias(
            "taxa_alfabetizacao"
        ),
        F.round(F.avg("inse_2021"), 3).alias("inse_medio"),
    )
    .orderBy("decil_inse")
)

## 2.8 Análise por Rede

In [0]:
display(
    df_inse_decis.groupBy("rede_descricao", "decil_inse")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.round(F.avg(F.col("alfabetizado").cast("double")) * 100, 2).alias(
            "taxa_alfabetizacao"
        ),
    )
    .orderBy("rede_descricao", "decil_inse")
)

## 2.9 Conclusão sobre o INSE

O INSE apresentou cobertura de aproximadamente 99,5% dos alunos da população de modelagem.

A análise exploratória indica associação entre contexto socioeconômico alfabetização. Os maiores níveis do indicador apresentam, em geral, maiores taxas de alfabetização, com diferença relevante entre as extremidades superiores e inferiores da distribuição.

A relação não é perfeitamente monotônica nos menores níveis do indicador, reforçando que o INSE representa apenas uma dimensão do problema e deve ser analisado em conjunto com características territoriais e educacionais.

O valor contínuo `inse_2021` será mantido como feature candidata para os modelos.

As faixas e decis foram utilizados apenas para análise exploratória.

# 3. Censo Escolar

O Censo Escolar será utilizado para representar características estruturais e
educacionais do ambiente escolar.

Como o identificador de escola presente na base de alfabetização é mascarado,
não é possível realizar a integração diretamente no nível da escola.

Por esse motivo, as características do Censo Escolar serão agregadas por
município e rede de ensino, granularidade compatível com a população de
modelagem.

Para preservar a relação temporal entre features e target, serão utilizados
dados do ano anterior à avaliação:

- Censo Escolar 2022 para a avaliação de alfabetização de 2023;
- Censo Escolar 2023 para a avaliação de alfabetização de 2024.

## 3.1 Carregamento da Base

A tabela de escolas do Censo Escolar é carregada diretamente da Base dos Dados
por meio do conector com o Google BigQuery.

Nesta primeira etapa, a estrutura da tabela é analisada para identificar as
variáveis potencialmente relevantes para o problema de alfabetização.

In [0]:
source_dataset_censo = "br_inep_censo_escolar"
source_table_censo = "escola"

tabela_censo = (
    f"{source_project}."
    f"{source_dataset_censo}."
    f"{source_table_censo}"
)

df_censo = (
    spark.read
    .format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load(tabela_censo)
)

df_censo.printSchema()

## 3.2 Cobertura Temporal

Antes da seleção das features, é verificada a disponibilidade de registros nos anos necessários para o enriquecimento da população de modelagem.

São utilizados os anos de 2022 e 2023, correspondentes ao período anterior às avaliações de alfabetização de 2023 e 2024.

In [0]:
display(
    df_censo
    .filter(F.col("ano").isin(2022, 2023))
    .groupBy("ano")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_escola").alias("qtd_escolas"),
        F.countDistinct("id_municipio").alias("qtd_municipios")
    )
    .orderBy("ano")
)

## 3.3 Compatibilidade Temporal e Recorte das Escolas

A análise inicial identificou uma diferença de disponibilidade entre as edições do Censo Escolar.

Em 2022, a variável específica de quantidade de matrículas no 2º ano não está preenchida. Por outro lado, a quantidade de matrículas nos anos iniciais do Ensino Fundamental está disponível tanto em 2022 quanto em 2023.

Para garantir comparabilidade temporal, são consideradas escolas que possuem matrículas nos anos iniciais do Ensino Fundamental.

O recorte mantém:

- anos de 2022 e 2023;
- redes Estadual e Municipal;
- escolas com pelo menos uma matrícula nos anos iniciais.

Dessa forma, as features utilizadas para as avaliações de 2023 e 2024 possuem uma definição consistente nos dois períodos.

In [0]:
df_censo_modelagem = df_censo.filter(
    (F.col("ano").isin(2022, 2023))
    & (F.col("rede").isin("2", "3"))
    & (F.col("quantidade_matricula_fundamental_anos_iniciais") > 0)
)

In [0]:
display(
    df_censo_modelagem
    .groupBy(
        "ano",
        "rede"
    )
    .agg(
        F.count("*").alias("qtd_escolas"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.sum("quantidade_matricula_fundamental_anos_iniciais")
        .alias("qtd_matriculas_anos_iniciais")
    )
    .orderBy(
        "ano",
        "rede"
    )
)

## 3.4 Seleção e Validação das Variáveis

O Censo Escolar possui centenas de atributos relacionados à estrutura e ao funcionamento das escolas.

Para evitar a inclusão excessiva de variáveis, foi selecionado um conjunto reduzido de características com interpretação direta para o contexto educacional dos alunos dos anos iniciais.

As features selecionadas representam três dimensões:

- infraestrutura básica;
- recursos educacionais e conectividade;
- estrutura de atendimento dos anos iniciais.

A seleção também considera a consistência do preenchimento entre 2022 e 2023. Variáveis com elevada proporção de valores ausentes foram retiradas da primeira versão da base de modelagem.

Após o recorte das escolas elegíveis, é avaliada a presença de valores nulos nas variáveis selecionadas antes da construção dos indicadores agregados.

In [0]:
variaveis_binarias_censo = [
    "agua_potavel",
    "energia_rede_publica",
    "esgoto_rede_publica",
    "biblioteca_sala_leitura",
    "laboratorio_informatica",
    "internet_aprendizagem"
]

variaveis_numericas_censo = [
    "quantidade_matricula_educacao_basica",
    "quantidade_matricula_fundamental_anos_iniciais",
    "quantidade_docente_fundamental_anos_iniciais",
    "quantidade_turma_fundamental_anos_iniciais"
]

variaveis_censo = (
    variaveis_binarias_censo
    + variaveis_numericas_censo
)

In [0]:
total_censo = df_censo_modelagem.count()

expressoes_nulos = [
    F.sum(F.when(F.col(coluna).isNull(), 1).otherwise(0)).alias(coluna)
    for coluna in variaveis_censo
]

resultado_nulos = df_censo_modelagem.agg(*expressoes_nulos).collect()[0]

nulos_censo = [
    (
        coluna,
        resultado_nulos[coluna],
        round(resultado_nulos[coluna] / total_censo * 100, 2),
    )
    for coluna in variaveis_censo
]

df_nulos_censo = spark.createDataFrame(
    nulos_censo, ["variavel", "qtd_nulos", "pct_nulos"]
)

display(df_nulos_censo.orderBy(F.desc("pct_nulos")))

## 3.5 Construção dos Indicadores Educacionais

Como o identificador individual das escolas não pode ser integrado à base de alfabetização, as características escolares são consolidadas por município e rede de ensino.

As proporções de infraestrutura são ponderadas pela quantidade de matrículas nos anos iniciais do Ensino Fundamental.

Dessa forma, escolas que atendem maior quantidade de alunos possuem maior peso na representação do contexto educacional do município e da rede.

Também são construídos indicadores relativos à quantidade de docentes e turmas dos anos iniciais.

In [0]:
df_censo_municipio_rede = df_censo_modelagem.groupBy("ano", "id_municipio", "rede").agg(
    # Quantidade de escolas consideradas
    F.countDistinct("id_escola").alias("censo_qtd_escolas"),
    # Matrículas dos anos iniciais
    F.sum("quantidade_matricula_fundamental_anos_iniciais").alias(
        "censo_qtd_matriculas_anos_iniciais"
    ),
    # Infraestrutura básica
    (
        F.sum(
            F.col("agua_potavel")
            * F.col("quantidade_matricula_fundamental_anos_iniciais")
        )
        / F.sum("quantidade_matricula_fundamental_anos_iniciais")
    ).alias("censo_pct_agua_potavel"),
    (
        F.sum(
            F.col("energia_rede_publica")
            * F.col("quantidade_matricula_fundamental_anos_iniciais")
        )
        / F.sum("quantidade_matricula_fundamental_anos_iniciais")
    ).alias("censo_pct_energia_rede_publica"),
    (
        F.sum(
            F.col("esgoto_rede_publica")
            * F.col("quantidade_matricula_fundamental_anos_iniciais")
        )
        / F.sum("quantidade_matricula_fundamental_anos_iniciais")
    ).alias("censo_pct_esgoto_rede_publica"),
    # Recursos educacionais
    (
        F.sum(
            F.col("biblioteca_sala_leitura")
            * F.col("quantidade_matricula_fundamental_anos_iniciais")
        )
        / F.sum("quantidade_matricula_fundamental_anos_iniciais")
    ).alias("censo_pct_biblioteca_sala_leitura"),
    (
        F.sum(
            F.col("laboratorio_informatica")
            * F.col("quantidade_matricula_fundamental_anos_iniciais")
        )
        / F.sum("quantidade_matricula_fundamental_anos_iniciais")
    ).alias("censo_pct_laboratorio_informatica"),
    (
        F.sum(
            F.col("internet_aprendizagem")
            * F.col("quantidade_matricula_fundamental_anos_iniciais")
        )
        / F.sum("quantidade_matricula_fundamental_anos_iniciais")
    ).alias("censo_pct_internet_aprendizagem"),
    # Estrutura de atendimento
    (
        F.sum("quantidade_docente_fundamental_anos_iniciais")
        / F.sum("quantidade_matricula_fundamental_anos_iniciais")
        * 100
    ).alias("censo_docentes_por_100_alunos"),
    (
        F.sum("quantidade_turma_fundamental_anos_iniciais")
        / F.sum("quantidade_matricula_fundamental_anos_iniciais")
        * 100
    ).alias("censo_turmas_por_100_alunos"),
)

## 3.6 Alinhamento Temporal

Os indicadores educacionais são utilizados de forma defasada em relação à avaliação de alfabetização.

Assim:

- Censo Escolar 2022 → avaliação de alfabetização 2023;
- Censo Escolar 2023 → avaliação de alfabetização 2024.

Essa estratégia garante que as informações utilizadas como features estejam disponíveis antes do período correspondente ao target.

In [0]:
df_censo_features = df_censo_municipio_rede.withColumn("ano", F.col("ano") + 1)

In [0]:
display(
    df_censo_features.agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct(F.struct("ano", "id_municipio", "rede")).alias("qtd_chaves"),
    )
)

## 3.7 Cobertura por Município e Rede

Inicialmente, a cobertura dos indicadores do Censo Escolar é avaliada nas combinações de ano, município e rede existentes na população de modelagem.

Essa análise permite identificar se os contextos educacionais utilizados pelo modelo possuem correspondência com os dados históricos do Censo.

In [0]:
contextos_modelagem = df_populacao_modelagem.select(
    "ano", "id_municipio", "rede"
).distinct()

cobertura_censo = contextos_modelagem.join(
    df_censo_features, on=["ano", "id_municipio", "rede"], how="left"
)


display(
    cobertura_censo.groupBy("ano")
    .agg(
        F.count("*").alias("qtd_contextos"),
        F.sum(F.when(F.col("censo_qtd_escolas").isNotNull(), 1).otherwise(0)).alias(
            "contextos_com_censo"
        ),
    )
    .withColumn(
        "pct_cobertura",
        F.round(F.col("contextos_com_censo") / F.col("qtd_contextos") * 100, 2),
    )
    .orderBy("ano")
)

## 3.8 Cobertura na População de Alunos

Após a validação territorial, os indicadores do Censo Escolar são associados aos registros individuais.

A cobertura final é calculada no nível dos alunos para verificar qual proporção da população de modelagem possui informações educacionais disponíveis.

In [0]:
df_populacao_com_censo = df_populacao_modelagem.join(
    df_censo_features, on=["ano", "id_municipio", "rede"], how="left"
)

display(
    df_populacao_com_censo.groupBy("ano")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.sum(F.when(F.col("censo_qtd_escolas").isNotNull(), 1).otherwise(0)).alias(
            "alunos_com_censo"
        ),
    )
    .withColumn(
        "pct_cobertura",
        F.round(F.col("alunos_com_censo") / F.col("qtd_alunos") * 100, 2),
    )
    .orderBy("ano")
)

## 3.9 Conclusão sobre o Censo Escolar

Os indicadores derivados do Censo Escolar apresentaram cobertura praticamente total da população de modelagem.

As variáveis selecionadas possuem preenchimento completo nos anos analisados e representam diferentes dimensões do contexto educacional, incluindo infraestrutura básica, recursos de aprendizagem e estrutura de atendimento.

A agregação por município e rede resultou em uma chave única e compatível com a base de alfabetização.

Os indicadores do Censo Escolar serão mantidos como features candidatas para a base analítica final, utilizando dados de 2022 para a avaliação de 2023 e dados de 2023 para a avaliação de 2024.

# 4. Contexto Econômico e Demográfico Municipal

Nesta etapa são incorporadas informações econômicas e demográficas dos municípios.

O objetivo é representar características territoriais que não estão presentes na base individual de alfabetização.

São utilizadas duas fontes do IBGE:

- Produto Interno Bruto municipal;
- população municipal.

A população permite representar o porte do município e também será utilizada na construção do PIB per capita, tornando o indicador econômico mais comparável entre municípios de diferentes tamanhos.

Para preservar a relação temporal entre features e target, serão utilizados:

- dados de 2022 para a avaliação de alfabetização de 2023;
- dados de 2023 para a avaliação de alfabetização de 2024.

### Limitação dos Componentes Setoriais do PIB

Os campos relacionados ao valor adicionado por atividade econômica não possuem dados disponíveis para 2022 e 2023.

Essa ausência não decorre de problema na ingestão. O IBGE não disponibilizou, para esses anos, a abertura municipal tradicional do valor adicionado por Agropecuária, Indústria, Serviços e Administração Pública.

Por esse motivo, os componentes setoriais foram retirados da primeira versão da base analítica.

O contexto econômico municipal será representado pelo PIB, população, PIB per capita e porte populacional em escala logarítmica.

## 4.1 Carregamento das Bases

As tabelas municipais de PIB e população são carregadas diretamente da Base dos Dados por meio do conector com o Google BigQuery.

Antes da construção das features, são avaliados os schemas e a cobertura temporal das duas fontes.

In [0]:
# PIB municipal
source_dataset_pib = "br_ibge_pib"
source_table_pib = "municipio"

tabela_pib = f"{source_project}.{source_dataset_pib}.{source_table_pib}"

df_pib = (
    spark.read.format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load(tabela_pib)
)

df_pib.printSchema()

In [0]:
# População municipal
source_dataset_populacao = "br_ibge_populacao"
source_table_populacao = "municipio"

tabela_populacao = (
    f"{source_project}.{source_dataset_populacao}.{source_table_populacao}"
)

df_populacao = (
    spark.read.format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load(tabela_populacao)
)

df_populacao.printSchema()

## 4.2 Cobertura Temporal

É verificada a disponibilidade de registros nos anos de 2022 e 2023, utilizados como referência anterior às avaliações de alfabetização.

In [0]:
display(
    df_pib.filter(F.col("ano").isin(2022, 2023))
    .groupBy("ano")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
    )
    .orderBy("ano")
)

In [0]:
display(
    df_populacao.filter(F.col("ano").isin(2022, 2023))
    .groupBy("ano")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
    )
    .orderBy("ano")
)

## 4.3 Integração de PIB e População

As bases de PIB e população possuem a mesma granularidade: município e ano.

As duas fontes são integradas por `ano` e `id_municipio`, permitindo a construção de indicadores econômicos comparáveis entre municípios de diferentes portes.

Além dos valores originais, serão derivados:

- PIB per capita, representando a atividade econômica em relação à população;
- logaritmo da população, reduzindo a assimetria entre municípios muito pequenos e grandes centros urbanos.

O PIB bruto e a população original serão preservados na base analítica para rastreabilidade, mas não necessariamente serão utilizados diretamente em todos os modelos.

In [0]:
df_contexto_municipal = (
    df_pib.filter(F.col("ano").isin(2022, 2023))
    .select("ano", "id_municipio", "pib")
    .join(
        df_populacao.filter(F.col("ano").isin(2022, 2023)).select(
            "ano", "id_municipio", "populacao"
        ),
        on=["ano", "id_municipio"],
        how="inner",
    )
    .withColumn(
        "pib_per_capita",
        F.when(F.col("populacao") > 0, F.col("pib") / F.col("populacao")),
    )
    .withColumn("log_populacao", F.log1p(F.col("populacao")))
    .withColumn("log_pib_per_capita", F.log1p(F.col("pib_per_capita")))
)

## 4.3 Validação da Qualidade

Antes da integração com a população de modelagem, é verificada a presença de valores ausentes nas features econômicas e demográficas construídas.

Também é validada a granularidade da base para garantir uma única linha por município e ano.

In [0]:
features_contexto = [
    "pib",
    "populacao",
    "pib_per_capita",
    "log_populacao",
    "log_pib_per_capita",
]

display(
    df_contexto_municipal.agg(
        *[
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in features_contexto
        ]
    )
)

## 4.4 Alinhamento Temporal

As informações econômicas e demográficas são utilizadas de forma defasada em relação à avaliação de alfabetização.

Dessa forma:

- dados municipais de 2022 são associados à avaliação de alfabetização de 2023;
- dados municipais de 2023 são associados à avaliação de alfabetização de 2024.

Essa estratégia garante que as características utilizadas como features sejam provenientes de um período anterior ao target.

In [0]:
df_contexto_features = df_contexto_municipal.withColumn("ano", F.col("ano") + 1)

display(
    df_contexto_features.agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct(F.struct("ano", "id_municipio")).alias("qtd_chaves"),
    )
)

## 4.7 Cobertura na População de Modelagem

Os indicadores econômicos e demográficos são associados à população de modelagem utilizando as chaves `ano` e `id_municipio`.

A cobertura é avaliada no nível dos alunos para verificar se os municípios presentes nas avaliações possuem informações econômicas e demográficas disponíveis.

In [0]:
df_populacao_com_contexto = df_populacao_modelagem.join(
    df_contexto_features, on=["ano", "id_municipio"], how="left"
)

display(
    df_populacao_com_contexto.groupBy("ano")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.sum(F.when(F.col("pib_per_capita").isNotNull(), 1).otherwise(0)).alias(
            "alunos_com_contexto"
        ),
    )
    .withColumn(
        "pct_cobertura",
        F.round(F.col("alunos_com_contexto") / F.col("qtd_alunos") * 100, 2),
    )
    .orderBy("ano")
)

## 4.8 Análise Exploratória

Após a validação da cobertura, é realizada uma análise exploratória da relação entre características econômicas e demográficas municipais e o target de alfabetização.

São analisados principalmente:

- PIB per capita, como indicador do contexto econômico municipal;
- população em escala logarítmica, como representação do porte do município.

As análises possuem caráter associativo e não devem ser interpretadas como relações causais.



In [0]:
display(
    df_populacao_com_contexto.filter(F.col("pib_per_capita").isNotNull())
    .groupBy("alfabetizado")
    .agg(
        F.round(F.avg("pib_per_capita"), 2).alias("pib_per_capita_medio"),
        F.round(F.expr("percentile_approx(pib_per_capita, 0.5)"), 2).alias(
            "pib_per_capita_mediano"
        ),
        F.round(F.avg("log_populacao"), 3).alias("log_populacao_medio"),
        F.round(F.avg("log_pib_per_capita"), 3).alias("log_pib_per_capita_medio"),
    )
    .orderBy("alfabetizado")
)

In [0]:
window_pib = Window.orderBy("pib_per_capita")

df_pib_decis = df_populacao_com_contexto.filter(
    F.col("pib_per_capita").isNotNull()
).withColumn("decil_pib_per_capita", F.ntile(10).over(window_pib))


display(
    df_pib_decis.groupBy("decil_pib_per_capita")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.round(F.avg(F.col("alfabetizado").cast("double")) * 100, 2).alias(
            "taxa_alfabetizacao"
        ),
        F.round(F.avg("pib_per_capita"), 2).alias("pib_per_capita_medio"),
    )
    .orderBy("decil_pib_per_capita")
)

In [0]:
window_populacao = Window.orderBy("populacao")

df_populacao_decis = df_populacao_com_contexto.filter(
    F.col("populacao").isNotNull()
).withColumn("decil_populacao", F.ntile(10).over(window_populacao))

display(
    df_populacao_decis.groupBy("decil_populacao")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.round(F.avg(F.col("alfabetizado").cast("double")) * 100, 2).alias(
            "taxa_alfabetizacao"
        ),
        F.round(F.avg("populacao"), 0).alias("populacao_media"),
    )
    .orderBy("decil_populacao")
)

## 4.9 Conclusão sobre o Contexto Econômico e Demográfico

As informações de PIB e população municipal apresentaram estrutura adequada para integração à população de modelagem.

O contexto econômico será representado principalmente pelo `pib_per_capita`, enquanto o porte municipal será representado por `log_populacao`.

Os valores originais de PIB e população serão preservados na base analítica para rastreabilidade e análises exploratórias.

Os indicadores serão associados temporalmente utilizando dados de 2022 para a avaliação de 2023 e dados de 2023 para a avaliação de 2024.